# Genie: Silver Table Questions

**Run on serverless compute. Executable version of the Anchor section of the slide deck (slides 6–7, plus validation slides).**

This notebook runs three anchor questions and two validation pairs as live before/after comparisons. Each before question asks against the base Silver tables using volume, frequency, or balance as a proxy. The after question asks the structurally equivalent question against the enriched Gold tables, where `community_id`, `fraud_risk_tier`, `is_ring_candidate`, and `risk_score` are available as ordinary columns.

- **Anchor 1 — Merchant Favorites** (primary slide pair): top-decile-by-volume merchants vs ring-candidate community merchants.
- **Anchor 2 — Book Share:** balance held by top-decile-by-volume vs balance held by ring-candidate communities.
- **Anchor 3 — Investigator Review Queue:** accounts in top decile by volume vs accounts at high risk tier, by region.
- **Validation A — Merchant Ring-Candidate Share:** which of four candidate merchants carry structural signal over the book baseline.
- **Validation B — High-Volume Account Community Membership:** the graph sometimes confirms the before reading — 19 of 20 top-volume accounts come back low risk.

The warm-up and analytics challenge confirm Genie is working and handling joins and conditional aggregates correctly before the anchors run.

In [ ]:
%pip install --upgrade databricks-sdk --quiet
dbutils.library.restartPython()

In [ ]:
CATALOG  = "graph-on-databricks"
SCHEMA   = "graph-enriched-schema"

SPACE_ID_BEFORE = dbutils.secrets.get("neo4j-graph-engineering", "genie_space_id_before")
SPACE_ID_AFTER  = dbutils.secrets.get("neo4j-graph-engineering", "genie_space_id_after")

In [ ]:
from databricks.sdk import WorkspaceClient
from demo_utils import genie_caller

w = WorkspaceClient()
print("Connected to:", w.config.host)
ask_before = genie_caller(w, SPACE_ID_BEFORE)
ask_after  = genie_caller(w, SPACE_ID_AFTER)

## Warm-Up: Tabular Baseline

Rank accounts by total spend. Confirms Genie is working and establishes the baseline before the anchor questions run.

In [ ]:
WARMUP_QUESTION = "What are the top 10 accounts by total amount spent across all merchants?"

print("[W] warm_up — asking...")
response_warmup = ask_before(WARMUP_QUESTION)

if response_warmup["df"] is not None:
    print(f"    Rows: {len(response_warmup['df'])}")
    print(f"    SQL:  {response_warmup['sql']}")
    display(response_warmup["df"])
else:
    print(f"    Status: {response_warmup['status']}")
    print(f"    Text:   {response_warmup['text']}")

## Analytics Challenge

Accounts with both elevated total spend and above-average night activity. Night transactions (hours 0–5) are an established fraud signal. Genie resolves this with a join and a conditional aggregate — all three dimensions exist in the base tables.

**Expected outcome:** a clean top-15 list with total spend, night ratio, and balance columns.

In [ ]:
ANALYTICS_QUESTION = (
    "Which accounts have both above-average total spend and a night transaction ratio "
    "above 20%? Show the top 15 by total spend with their night ratio and account balance."
)

print("[A] analytics_challenge — asking...")
response_analytics = ask_before(ANALYTICS_QUESTION)

if response_analytics["df"] is not None:
    print(f"    Rows: {len(response_analytics['df'])}")
    print(f"    SQL:  {response_analytics['sql']}")
    display(response_analytics["df"])
else:
    print(f"    Status: {response_analytics['status']}")
    print(f"    Text:   {response_analytics['text']}")

## Anchor Questions

Three questions, each run twice — once against the Silver tables (before), once against the enriched Gold tables (after). The before question uses a volume or frequency proxy. The after question uses `community_id`, `fraud_risk_tier`, or `risk_score` as the structural signal the proxy was approximating. The first question also includes a follow-up that drills into the communities the after answer surfaces.

---

### 1. Merchant Favorites

**Before:** which merchants do the top 10% of accounts by volume visit most? Volume is the only available proxy for ring membership on the Silver tables.

**After:** which merchants are most commonly visited by accounts in ring-candidate communities? Not merchants that are popular overall, but merchants where ring members cluster disproportionately.

**Follow-up:** which ring-candidate communities have the highest total transaction volume, and how many member accounts do they contain? Drills from the merchant signal to the communities behind it.

In [ ]:
ANCHOR1_BEFORE = (
    "Which merchants are most commonly visited by the top 10% of accounts "
    "by total transaction volume?"
)

print("[1B] merchant_favorites — before (Silver)] asking...")
r1b = ask_before(ANCHOR1_BEFORE)

if r1b["df"] is not None:
    print(f"    Rows: {len(r1b['df'])}")
    print(f"    SQL:  {r1b['sql']}")
    display(r1b["df"])
else:
    print(f"    Status: {r1b['status']}")
    print(f"    Text:   {r1b['text']}")

In [ ]:
ANCHOR1_AFTER = (
    "Which merchants are most commonly visited by accounts in ring-candidate communities?"
)

print("[1A] merchant_favorites — after (Gold)] asking...")
r1a = ask_after(ANCHOR1_AFTER)

if r1a["df"] is not None:
    print(f"    Rows: {len(r1a['df'])}")
    print(f"    SQL:  {r1a['sql']}")
    display(r1a["df"])
else:
    print(f"    Status: {r1a['status']}")
    print(f"    Text:   {r1a['text']}")

In [ ]:
ANCHOR1_FOLLOWUP = (
    "Which ring-candidate communities have the highest total transaction volume, "
    "and how many member accounts do they contain?"
)

print("[1F] merchant_favorites — follow-up (Gold)] asking...")
r1f = ask_after(ANCHOR1_FOLLOWUP)

if r1f["df"] is not None:
    print(f"    Rows: {len(r1f['df'])}")
    print(f"    SQL:  {r1f['sql']}")
    display(r1f["df"])
else:
    print(f"    Status: {r1f['status']}")
    print(f"    Text:   {r1f['text']}")

### 2. Book Share

**Before:** for the top 10% of accounts by transfer volume, what is the total balance held and what share of the book do they represent? Volume decile is the proxy for ring membership.

**After:** for ring-candidate communities taken together, what is the total balance held by their members and what share of the book do they represent? That is the number risk leadership asks for.

In [ ]:
ANCHOR2_BEFORE = (
    "For the top 10% of accounts by transfer volume, what is the total balance "
    "held and what share of the book do they represent?"
)

print("[2B] book_share — before (Silver)] asking...")
r2b = ask_before(ANCHOR2_BEFORE)

if r2b["df"] is not None:
    print(f"    Rows: {len(r2b['df'])}")
    print(f"    SQL:  {r2b['sql']}")
    display(r2b["df"])
else:
    print(f"    Status: {r2b['status']}")
    print(f"    Text:   {r2b['text']}")

In [ ]:
ANCHOR2_AFTER = (
    "For ring-candidate communities taken together, what is the total balance "
    "held by their members and what share of the book do they represent?"
)

print("[2A] book_share — after (Gold)] asking...")
r2a = ask_after(ANCHOR2_AFTER)

if r2a["df"] is not None:
    print(f"    Rows: {len(r2a['df'])}")
    print(f"    SQL:  {r2a['sql']}")
    display(r2a["df"])
else:
    print(f"    Status: {r2a['status']}")
    print(f"    Text:   {r2a['text']}")

### 3. Investigator Review Queue

**Before:** how many accounts are in the top 10% by transfer volume, and what is the regional breakdown? Volume cutoff is the only available threshold for sizing an investigation workload.

**After:** how many accounts would need investigator review if the bar is high risk tier, and what is the regional breakdown? A risk manager immediately sees the difference — volume cutoff versus a defensible structural segment.

In [ ]:
ANCHOR3_BEFORE = (
    "How many accounts are in the top 10% by transfer volume, "
    "and what is the regional breakdown of that workload?"
)

print("[3B] review_queue — before (Silver)] asking...")
r3b = ask_before(ANCHOR3_BEFORE)

if r3b["df"] is not None:
    print(f"    Rows: {len(r3b['df'])}")
    print(f"    SQL:  {r3b['sql']}")
    display(r3b["df"])
else:
    print(f"    Status: {r3b['status']}")
    print(f"    Text:   {r3b['text']}")

In [ ]:
ANCHOR3_AFTER = (
    "How many accounts would need investigator review if the bar is high risk tier, "
    "and what is the regional breakdown of that workload?"
)

print("[3A] review_queue — after (Gold)] asking...")
r3a = ask_after(ANCHOR3_AFTER)

if r3a["df"] is not None:
    print(f"    Rows: {len(r3a['df'])}")
    print(f"    SQL:  {r3a['sql']}")
    display(r3a["df"])
else:
    print(f"    Status: {r3a['status']}")
    print(f"    Text:   {r3a['text']}")

---

## Validation Pairs

Two more before/after pairs from the slide deck's validation section. Use these after the core anchors to show that the graph can validate as well as surface — sometimes it flags a structural outlier the proxy missed, and sometimes it confirms the proxy's reading rather than overturning it.

---

### Validation A: Merchant Ring-Candidate Share

**Before:** which merchants are most commonly visited by the top 20 accounts by transaction volume? The volume proxy returns a flat, dispersed list — 243 merchants with no co-visit count above 2, nothing stands out.

**After:** for **James-Conway**, **Cardenas and Sons**, **Johnson, Williams and May**, and **Meyer Ltd**, what share of each merchant's customers are in ring-candidate communities versus the ~4% book baseline? Three sit at baseline; **James-Conway** (crypto) is ~19× above it. The before could not distinguish the outlier from the noise; the after can.

In [ ]:
VALIDATION1_BEFORE = (
    "Which merchants are most commonly visited by the top 20 accounts "
    "by total transaction volume?"
)

print("[V1B] merchant_ring_share — before (Silver)] asking...")
rv1b = ask_before(VALIDATION1_BEFORE)

if rv1b["df"] is not None:
    print(f"    Rows: {len(rv1b['df'])}")
    print(f"    SQL:  {rv1b['sql']}")
    display(rv1b["df"])
else:
    print(f"    Status: {rv1b['status']}")
    print(f"    Text:   {rv1b['text']}")

In [ ]:
VALIDATION1_AFTER = (
    "For James-Conway, Cardenas and Sons, Johnson, Williams and May, and Meyer Ltd, "
    "what share of each merchant's customers are members of ring-candidate "
    "communities, and how does that compare to the book baseline?"
)

print("[V1A] merchant_ring_share — after (Gold)] asking...")
rv1a = ask_after(VALIDATION1_AFTER)

if rv1a["df"] is not None:
    print(f"    Rows: {len(rv1a['df'])}")
    print(f"    SQL:  {rv1a['sql']}")
    display(rv1a["df"])
else:
    print(f"    Status: {rv1a['status']}")
    print(f"    Text:   {rv1a['text']}")

### Validation B: High-Volume Account Community Membership

**Before:** for the top 20 accounts by transaction volume, how many unique merchants did each visit? The answer looks like legitimate diverse spending — 7 to 21 merchants per account, no correlation with volume.

**After:** what is their community membership and risk tier? The graph confirms the before reading — 19 of 20 accounts are low risk, concentrated in three known low-risk communities. The enrichment sometimes exonerates; not every before/after gap flips the interpretation.

In [ ]:
VALIDATION2_BEFORE = (
    "For the top 20 accounts by total transaction volume, how many unique "
    "merchants did each account visit?"
)

print("[V2B] top20_account_membership — before (Silver)] asking...")
rv2b = ask_before(VALIDATION2_BEFORE)

if rv2b["df"] is not None:
    print(f"    Rows: {len(rv2b['df'])}")
    print(f"    SQL:  {rv2b['sql']}")
    display(rv2b["df"])
else:
    print(f"    Status: {rv2b['status']}")
    print(f"    Text:   {rv2b['text']}")

In [ ]:
VALIDATION2_AFTER = (
    "For accounts in the top 20 by total transaction volume, what is their "
    "community membership status and risk tier? Are those accounts concentrated "
    "in a small number of communities, or are they spread across the book?"
)

print("[V2A] top20_account_membership — after (Gold)] asking...")
rv2a = ask_after(VALIDATION2_AFTER)

if rv2a["df"] is not None:
    print(f"    Rows: {len(rv2a['df'])}")
    print(f"    SQL:  {rv2a['sql']}")
    display(rv2a["df"])
else:
    print(f"    Status: {rv2a['status']}")
    print(f"    Text:   {rv2a['text']}")

In [ ]:
def _status(r):
    if r["df"] is not None and len(r["df"]) > 0:
        return "RESPONDED"
    elif r["df"] is not None:
        return "NO DATA"
    return "NO DATA" if r["text"] else "ERROR"

results = [
    ("merchant_favorites         before",    r1b),
    ("merchant_favorites         after",     r1a),
    ("merchant_favorites         follow-up", r1f),
    ("book_share                 before",    r2b),
    ("book_share                 after",     r2a),
    ("review_queue               before",    r3b),
    ("review_queue               after",     r3a),
    ("merchant_ring_share        before",    rv1b),
    ("merchant_ring_share        after",     rv1a),
    ("top20_account_membership   before",    rv2b),
    ("top20_account_membership   after",     rv2a),
]

print("=" * 78)
print("Genie Comparison — Summary")
print("=" * 78)
print()
for name, r in results:
    print(f"  {_status(r):<12}  {name}")
print()
print("-" * 78)
print("The before answers use volume proxies. The after answers use community_id,")
print("fraud_risk_tier, risk_score, and is_ring_candidate. The gap between each")
print("pair is the demo's argument. Validations show the graph can surface AND")
print("exonerate — Validation A flags merchant_4859; Validation B confirms the")
print("before reading that 19 of 20 top-volume accounts are low risk.")

## Next Steps

Proceed to **`02_neo4j_ingest`** to load the base tables into Neo4j as a property graph. After the full pipeline (`02_neo4j_ingest` → `03_gds_enrichment` → `04_pull_gold_tables`), the Gold Genie space will be populated and the after questions will return structurally different results. Use `genie-guide.md` for the copy-paste versions of all questions.